# Train word2vec

In [ ]:
"""Test Function"""
import os
from gensim.models import Word2Vec

# 1. Đọc và unique tokens
def read_unique_tokens(folder_path):
    tokens = []
    for fname in os.listdir(folder_path):
        path = os.path.join(folder_path, fname)
        if not os.path.isfile(path):
            continue
        with open(path, encoding='utf-8') as f:
            for line in f:
                if "TOKENIZED" in line:
                    parts = line.partition(":")[2].strip().split()
                    tokens.append(parts)
    # loại trùng giữ thứ tự
    return tokens

# Thư mục dữ liệu và model
base_folder  = ["data_c", "data_cpp", "data_java"]
for folder in base_folder:
    folder_token = os.path.join(folder, "tokenized_contexts")
    model_folder = os.path.join(folder, "model")
    os.makedirs(model_folder, exist_ok=True)

    tokens = read_unique_tokens(folder_token)

    # 2. Huấn luyện Word2Vec (CBOW)
    model = Word2Vec(
        sentences= tokens ,
        vector_size=512,
        window=5,
        min_count=1,
        workers=4,
        sg=0,      # 0 = CBOW, 1 = Skip-gram
        epochs=5   # Tăng nếu cần
    )

    # 3. Lưu model
    model_path = os.path.join(model_folder, f"word2vec.model")
    model.save(model_path)
    print(f"Đã lưu Word2Vec model vào: {model_path}")

# Subgraph embedding

In [1]:
import os, re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from extractToken import extract_and_replace_tokens
import torch
from torch_geometric.data import Data

# ———————— Cấu hình đường dẫn ————————
BASE_FOLDERS = ["data_java", "data_c", "data_cpp"]

# ———————— 1. Hàm parse 1 file context ————————
def parse_context_file(path):
    """
    Đọc file context, gom multi-line CODE[...] về một dòng,
    rồi parse như bình thường.
    """
    # 1) Load và gộp các dòng node/edge multi-line
    flattened = []
    buffer = None
    with open(path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.rstrip('\n')
            # Nếu đang trong buffer (đã bắt đầu một block nhưng chưa kết thúc)
            if buffer is not None:
                buffer += ' ' + line.strip()
                if line.strip().endswith('];'):
                    flattened.append(buffer)
                    buffer = None
                continue

            # Nếu là dòng bắt đầu node/edge
            if re.match(r'^\s*"\d+"\s*\[', line) and not line.strip().endswith('];'):
                # mở buffer và tiếp tục đọc
                buffer = line.strip()
                continue

            # bình thường: thêm thẳng
            flattened.append(line)

    # 2) Dùng flattened list thay cho việc đọc line-by-line
    subgraphs = []
    current = None
    attr_pattern = re.compile(r'(\w+)="([^"]*)"')

    for line in flattened:
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        # START_SUBGRAPH
        m_start = re.match(r'^START_SUBGRAPH center_node=(\d+)', line)
        if m_start:
            center = m_start.group(1)
            current = {'center_str': center, 'nodes': [], 'edges': []}
            subgraphs.append(current)
            continue

        # END_SUBGRAPH
        if line.startswith('END_SUBGRAPH'):
            current = None
            continue

        if current is None:
            continue

        # edge
        if '->' in line:
            m = re.match(
                r'^"(?P<src>\d+)"\s*->\s*"(?P<dst>\d+)"\s*\[(?P<attrs>.+)\];', line
            )
            if m:
                attrs = dict(attr_pattern.findall(m.group('attrs')))
                current['edges'].append({
                    'src_str': m.group('src'),
                    'dst_str': m.group('dst'),
                    'type': attrs.get('label')
                })
            continue

        # node
        m = re.match(r'^"(?P<id>\d+)"\s*\[(?P<attrs>.+)\];', line)
        if m:
            attrs = dict(attr_pattern.findall(m.group('attrs')))
            current['nodes'].append({
                'id_str': m.group('id'),
                'label': attrs.get('label'),
                'name': attrs.get('NAME'),
                'code': attrs.get('CODE')
            })
            continue

    # 3) Sắp xếp và đánh id như cũ
    for sg in subgraphs:
        center_str = sg['center_str']
        # đảm bảo có center node
        center_nodes = [n for n in sg['nodes'] if n['id_str']==center_str]
        if not center_nodes:
            # thêm node center trống nếu cần
            sg['nodes'].append({'id_str': center_str, 'label':None, 'name':None, 'code':None})
            center_node = sg['nodes'][-1]
        else:
            center_node = center_nodes[0]

        others = [n for n in sg['nodes'] if n['id_str']!=center_str]
        ordered = [center_node] + others

        id_map = {}
        for idx, node in enumerate(ordered):
            node['id'] = idx
            id_map[node['id_str']] = idx
        sg['nodes'] = ordered

        real_edges = []
        for e in sg['edges']:
            if e['src_str'] in id_map and e['dst_str'] in id_map:
                real_edges.append({
                    'src': id_map[e['src_str']],
                    'dst': id_map[e['dst_str']],
                    'type': e['type']
                })
        sg['edges'] = real_edges
        del sg['center_str']

    return subgraphs


# ———————— 2. Load mapping node-type ————————
def load_type_mapping(csv_path):
    df = pd.read_csv(csv_path)
    mapping = {}
    for idx, nt in enumerate(df['Node Type'].astype(str)):
        for key in re.split(r'\s*/\s*', nt):
            if key:
                mapping[key] = idx
    return mapping

# ———————— 3. Classify nodes & edges ————————
def classify_nodes(subgraphs, mapping):
    for sg in subgraphs:
        for node in sg['nodes']:
            t = None
            lbl = node.get('label')
            if lbl and lbl in mapping:
                t = mapping[lbl]
            else:
                nm = node.get('name') or ''
                key = nm.split('.',1)[1] if '.' in nm else nm
                t = mapping.get(key)
            node['type_id'] = t if t is not None else -1
    return subgraphs

def classify_edges(subgraphs):
    for sg in subgraphs:
        for e in sg['edges']:
            tt = (e.get('type') or '').strip().lower()
            e['type_id'] = 1 if tt=='cfg' else 2 if tt=='ast' else 0
    return subgraphs

# ———————— 4. Tính code_vector & feature_vector ————————
def compute_code_vector(code: str, w2v: Word2Vec, vec_size: int):
    code = code or ""
    toks = extract_and_replace_tokens(code).split()
    vecs = [w2v.wv[t] for t in toks if t in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(vec_size,)

def add_vectors(subgraphs, w2v: Word2Vec):
    vec_size = w2v.vector_size
    for sg in subgraphs:
        for n in sg['nodes']:
            cv = compute_code_vector(n.get('code',''), w2v, vec_size)
            n['code_vector'] = cv.tolist()
            # concat type_id trước
            fv = np.concatenate(([n.get('type_id',0)], cv))
            n['feature_vector'] = fv.tolist()
    return subgraphs

def add_feature_matrix(subgraphs, vec_size):
    for sg in subgraphs:
        # Collect all feature vectors from nodes
        features = [n['feature_vector'] for n in sg['nodes']]
        # Concatenate along axis 0 (rows = nodes, cols = features)
        feature_matrix = np.stack(features, axis=0) if features else np.zeros((0, 1 + vec_size))
        sg['feature_matrix'] = feature_matrix
    return subgraphs

# ———————— 5. Main loop: process & save ————————
def process_folder(base_folder):
    CTX_FOLDER = os.path.join(base_folder, "subgraph_contexts")
    OUT_FOLDER = os.path.join(base_folder, "processed_subgraphs")
    MODEL_PATH = os.path.join(base_folder, "model", "word2vec.model")
    CHAR_CSV = os.path.join(base_folder, "vuln-char-table-final.csv")
    PYG_DATA_FILE = os.path.join(OUT_FOLDER, "all_subgraphs_pyg.pt")

    os.makedirs(OUT_FOLDER, exist_ok=True)
    node_map = load_type_mapping(CHAR_CSV)
    w2v = Word2Vec.load(MODEL_PATH)
    vec_size = w2v.vector_size

    all_pyg_data = []
    skipped_graphs_count = 0

    for fname in os.listdir(CTX_FOLDER):
        path_in = os.path.join(CTX_FOLDER, fname)
        if not os.path.isfile(path_in):
            continue

        # 1) parse
        subs = parse_context_file(path_in)
        # 2) classify
        subs = classify_nodes(subs, node_map)
        subs = classify_edges(subs)
        # 3) vectors
        subs = add_vectors(subs, w2v)

        # Determine label from filename
        label = 0 if 'good' in fname or 'mixed' in fname else 1

        # Convert each subgraph dict to a PyG Data object and add to the list
        for i, sg_dict in enumerate(subs):
            # Calculate feature_matrix for the current subgraph
            features = [n['feature_vector'] for n in sg_dict['nodes']]
            feature_matrix = np.stack(features, axis=0) if features else np.empty((0, 1 + vec_size))

            # Validation: Check if graph has nodes
            if feature_matrix.shape[0] == 0:
                skipped_graphs_count += 1
                continue # Skip this subgraph if it has no nodes

            # Node features (x)
            x = torch.tensor(feature_matrix, dtype=torch.float)
            num_nodes = x.shape[0] # Get actual number of nodes

            # Edges (edge_index, edge_type)
            if sg_dict.get('edges') and len(sg_dict['edges']) > 0:
                # Filter edges again to ensure indices are valid *after* node processing
                valid_edges = [e for e in sg_dict['edges'] if e['src'] < num_nodes and e['dst'] < num_nodes]
                
                if valid_edges:
                    edge_index = torch.tensor(
                        [[e['src'], e['dst']] for e in valid_edges],
                        dtype=torch.long
                    ).t().contiguous()
                    edge_type = torch.tensor(
                        [e.get('type_id', 0) for e in valid_edges], 
                        dtype=torch.long
                    )
                else: # No valid edges remain
                    edge_index = torch.empty((2, 0), dtype=torch.long)
                    edge_type = torch.empty((0,), dtype=torch.long)
            else:
                # Handle cases with originally no edges
                edge_index = torch.empty((2, 0), dtype=torch.long)
                edge_type = torch.empty((0,), dtype=torch.long)

            # Label (y)
            y = torch.tensor([label], dtype=torch.long)
            
            # Create Data object
            data = Data(x=x, edge_index=edge_index, edge_type=edge_type, y=y)
            
            # Final Sanity Check (optional but recommended)
            if data.edge_index.numel() > 0 and data.edge_index.max().item() >= data.num_nodes:
                print(f"  [CRITICAL WARN] Graph {i} in {fname}: Detected invalid edge index ({data.edge_index.max().item()}) vs num_nodes ({data.num_nodes}) AFTER filtering. Skipping.")
                skipped_graphs_count += 1
                continue
                 
            all_pyg_data.append(data)

        print(f"Processed {fname}: Added {len(subs) - skipped_graphs_count} subgraphs (skipped {skipped_graphs_count} empty). Total added: {len(all_pyg_data)}")

    # Save the consolidated list of PyG Data objects using torch.save
    torch.save(all_pyg_data, PYG_DATA_FILE) 
    print(f"Total skipped graphs due to no nodes: {skipped_graphs_count}") # Print total skipped count
    print(f"All valid PyG Data objects ({len(all_pyg_data)} total) saved to {PYG_DATA_FILE}")

if __name__ == "__main__":
    for folder in BASE_FOLDERS:
        print(f"\nProcessing {folder}...")
        process_folder(folder)


Processing data_java...
Processed 250735-v1.0.0-mixed_context.txt: Added 34 subgraphs (skipped 0 empty). Total added: 34
Processed 155740-v1.0.0-bad_context.txt: Added 17 subgraphs (skipped 0 empty). Total added: 51
Processed 155183-v1.0.0-bad_context.txt: Added 42 subgraphs (skipped 0 empty). Total added: 93
Processed 156403-v1.0.0-bad_context.txt: Added 33 subgraphs (skipped 0 empty). Total added: 126
Processed 155758-v1.0.0-bad_context.txt: Added 49 subgraphs (skipped 0 empty). Total added: 175
Processed 251074-v1.0.0-mixed_context.txt: Added 10 subgraphs (skipped 0 empty). Total added: 185
Processed 251556-v1.0.0-mixed_context.txt: Added 8 subgraphs (skipped 0 empty). Total added: 193
Processed 251356-v1.0.0-mixed_context.txt: Added 8 subgraphs (skipped 0 empty). Total added: 201
Processed 250997-v1.0.0-mixed_context.txt: Added 26 subgraphs (skipped 0 empty). Total added: 227
Processed 155917-v1.0.0-bad_context.txt: Added 59 subgraphs (skipped 0 empty). Total added: 286
Processed 

# Train

In [2]:
# train.py

import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader, Dataset
from torch_geometric.nn import RGCNConv, global_mean_pool

# ——— 1) Thông số ———
PKL_FOLDER  = "data_java/processed_subgraphs"
PYG_DATA_FILE = os.path.join(PKL_FOLDER, "all_subgraphs_pyg.pt")
BATCH_SIZE  = 32
LR          = 1e-3
EPOCHS      = 50
RANDOM_SEED = 42
# New hyperparameters for regularization
DROPOUT_RATE = 0.5 # Example dropout rate
WEIGHT_DECAY = 5e-4 # Example weight decay (L2 regularization)
EARLY_STOPPING_PATIENCE = 10 # Example patience for early stopping

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ——— 2) Dataset flatten subgraphs ———
class SubgraphDataset(Dataset):
    def __init__(self, pyg_data_path, transform=None, pre_transform=None):
        super().__init__(None, transform, pre_transform)
        
        # Load the list of PyG Data objects directly from the .pt file
        if not os.path.exists(pyg_data_path):
            raise FileNotFoundError(f"PyG data file not found: {pyg_data_path}. "
                                    f"Please ensure '{pyg_data_path}' exists by running the preprocessing script.")

        # Load the list of Data objects using torch.load()
        print(f"Loading PyG data from {pyg_data_path}...") # Add loading message
        self.graphs = torch.load(pyg_data_path, weights_only=False)
        print(f"Finished loading {len(self.graphs)} graphs.")

        # --- Validation Step --- 
        print("Validating graph data integrity...")
        invalid_indices = []
        for i, data in enumerate(self.graphs):
            if data.edge_index is not None and data.edge_index.numel() > 0:
                num_nodes = data.num_nodes # Get number of nodes for this graph
                max_edge_index = data.edge_index.max().item()
                if max_edge_index >= num_nodes:
                    print(f"  [ERROR] Graph {i}: Invalid edge_index. Max index is {max_edge_index}, but num_nodes is {num_nodes}. Edge Index: {data.edge_index}")
                    invalid_indices.append(i)
            # Optional: Check for NaNs or Infs in features if needed
            # if torch.isnan(data.x).any() or torch.isinf(data.x).any():
            #    print(f"  [WARN] Graph {i}: Contains NaN/Inf in node features.")

        if invalid_indices:
             # Option 1: Raise error
             raise ValueError(f"Found {len(invalid_indices)} graphs with invalid edge indices (max index >= num_nodes). Check preprocessing script. Problematic graph indices: {invalid_indices[:10]}...")
             # Option 2: Filter out invalid graphs (use with caution)
             # print(f"[WARN] Removing {len(invalid_indices)} graphs with invalid edge indices.")
             # self.graphs = [g for i, g in enumerate(self.graphs) if i not in invalid_indices]
             # print(f"Dataset size after filtering: {len(self.graphs)}")
        else:
            print("Graph data validation successful.")
        # --- End Validation --- 

    def len(self): 
        return len(self.graphs)

    def get(self, idx):
        # Simply return the pre-loaded Data object
        return self.graphs[idx] 


# reproducibility
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# load dataset
dataset = SubgraphDataset(PYG_DATA_FILE)
print(f"Loaded {len(dataset)} graphs from {PYG_DATA_FILE}")

# ——— 3) Split train/val/test ———
all_idx = list(range(len(dataset)))
labels  = [dataset[i].y.item() for i in all_idx]

train_idx, test_idx = train_test_split(
    all_idx, test_size=0.2, random_state=RANDOM_SEED, stratify=labels
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.1, random_state=RANDOM_SEED,
    stratify=[labels[i] for i in train_idx]
)

train_loader = DataLoader(dataset[train_idx], batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(dataset[val_idx],   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(dataset[test_idx],  batch_size=BATCH_SIZE, shuffle=False)

print(f"Split sizes → train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")

# ——— 4) Model RGCN + MLP ———
class RGCN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, num_relations, num_classes, dropout_rate):
        super().__init__()
        self.conv1 = RGCNConv(in_dim,  hidden_dim, num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations)
        self.lin   = torch.nn.Linear(hidden_dim, num_classes)
        self.dropout = torch.nn.Dropout(p=dropout_rate) # Add dropout layer

    def forward(self, data):
        x, edge_index, edge_type, batch = (
            data.x, data.edge_index, data.edge_type, data.batch
        )
        x = F.relu(self.conv1(x, edge_index, edge_type))
        x = self.dropout(x) # Apply dropout
        x = F.relu(self.conv2(x, edge_index, edge_type))
        x = self.dropout(x) # Apply dropout
        x = global_mean_pool(x, batch)
        return self.lin(x)

# khởi tạo mô hình
sample      = dataset[0]
in_dim      = sample.num_node_features
num_rel     = 3 # Hardcoded based on the 3 types (0, 1, 2) defined in preprocessing
num_classes = int(sample.y.max().item()) + 1

model     = RGCN(in_dim, hidden_dim=128, num_relations=num_rel, 
                 num_classes=num_classes, dropout_rate=DROPOUT_RATE).to(device) # Pass dropout_rate
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY) # Add weight_decay

# ——— 5) Train & Validate ———
def train_epoch():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch)
        loss = F.cross_entropy(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_loader.dataset)

def evaluate(loader):
    model.eval()
    all_y, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out  = model(batch)
            pred = out.argmax(dim=1)
            all_y.append(batch.y.cpu().numpy())
            all_pred.append(pred.cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    return classification_report(y_true, y_pred, digits=4)

best_val_loss = float('inf') # Renamed for clarity
best_epoch = 0 # New: Track best epoch
patience_counter = 0 # New: Counter for early stopping

print("\nStarting training...")
for epoch in range(1, EPOCHS+1):
    train_loss = train_epoch()

    # --- Validation --- 
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out  = model(batch)
            val_loss += F.cross_entropy(out, batch.y).item() * batch.num_graphs
    val_loss /= len(val_loader.dataset)
    # --- End Validation ---

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # --- Checkpointing and Early Stopping Logic --- 
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch # Store the epoch number
        torch.save(model.state_dict(), "best_rgcn.pt")
        print(f"  Saved new best model (Epoch {best_epoch})")
        patience_counter = 0 # Reset patience counter
    else:
        patience_counter += 1 # Increment patience counter
        print(f"  Validation loss did not improve. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}. Best validation loss was {best_val_loss:.4f} at epoch {best_epoch}.")
        break # Exit the training loop
    # --- End Checkpointing --- 

# Ensure best_epoch is set if training finishes normally without early stopping
if patience_counter < EARLY_STOPPING_PATIENCE:
     print(f"\nTraining finished after {EPOCHS} epochs. Best validation loss was {best_val_loss:.4f} at epoch {best_epoch}.")

# ——— 6) Test ———
# Load the *best* model for testing
print(f"\nLoading best model from epoch {best_epoch} (Validation Loss: {best_val_loss:.4f}) for testing...")
model.load_state_dict(torch.load("best_rgcn.pt"))
print("=== Test Report ===")
print(evaluate(test_loader))

Using device: cuda
Loading PyG data from data_java/processed_subgraphs/all_subgraphs_pyg.pt...
Finished loading 141622 graphs.
Validating graph data integrity...
Graph data validation successful.
Loaded 141622 graphs from data_java/processed_subgraphs/all_subgraphs_pyg.pt


/home/keanlt/ML-Project/.venv/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Split sizes → train: 101967, val: 11330, test: 28325

Starting training...


/pytorch/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [2,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [4,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/Loss.cu:242: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [6,0,0] Assertion `t >= 0 && t < n_classes` failed.
/pytorch/aten/sr

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
